# 01 — Token Visualization (Diagnostic D1)

**Goal:** score every visual token against the question, then overlay the scores as a heatmap on the image.

**Diagnostic D1:** *do the high-scored tokens correspond to the objects mentioned in the question?* If the question asks about a car, the car's patches should light up. If the heatmap is diffuse or highlights the sky/background, the text-conditioning signal is weak.

**Location note:** scores are computed on **pre-decoder** connector output (81 tokens for SmolVLM at 384px) — never on post-decoder `last_hidden_state`.

In [ ]:
!pip -q install torch transformers accelerate Pillow requests
import sys, os, torch
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from transformers import AutoProcessor, AutoModelForImageTextToText
from tgtp.models.smolvlm_wrapper import SmolVLMPruningWrapper
from tgtp.utils.visualization import overlay_token_heatmap

MODEL = "HuggingFaceTB/SmolVLM-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
processor = AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(MODEL, torch_dtype=dtype, trust_remote_code=True).to(device)
model.eval()
wrapper = SmolVLMPruningWrapper(model)

In [ ]:
import requests
from PIL import Image
from io import BytesIO

url = "https://images.unsplash.com/photo-1502877338535-766e1452684a?w=600"
img = Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")
img.thumbnail((384, 384))
img

In [ ]:
def relevance_heatmap(image, question, grid=(9, 9)):
    pv = processor(images=image, return_tensors="pt")["pixel_values"].to(device)
    enc = processor(text=question, return_tensors="pt")
    input_ids, mask = enc["input_ids"].to(device), enc["attention_mask"].to(device)
    with torch.no_grad():
        visual = wrapper.extract_visual_tokens(pv)
        scores = wrapper.score(visual, input_ids, mask, scoring_method="cross_attention")
    scores = scores[0].cpu().numpy()
    path = f"heatmap_{question[:10].replace(' ', '_')}.png"
    overlay_token_heatmap(image, scores, grid[0], grid[1], path)
    return Image.open(path), scores

heat, scores = relevance_heatmap(img, "What color is the car?")
display(heat)
print("score stats:", scores.min().round(3), scores.max().round(3), scores.mean().round(3))

**D1 verdict:** check that the bright region overlaps the car. Run the same image with a different question (e.g. "What is the weather?") and observe that a *different* set of tokens lights up — that is the text-conditioning property the thesis relies on.

In [ ]:
heat2, scores2 = relevance_heatmap(img, "Is it sunny outside?")
display(heat2)
print("different prompt -> different scores?", not (scores == scores2).all())